# Épreuve Technique ADRIA / ENSET 2027
## Partie H — IA avec Python : Détection de Fraude Bancaire

Ce notebook présente la démarche complète de classification binaire pour la détection de transactions frauduleuses :
1. Chargement des données `bank_transactions.csv`
2. Exploration : proportion de fraudes et valeurs manquantes
3. Séparation entraînement/test stratifiée
4. Exclusion des identifiants (`transaction_id`, `account_id`)
5. Construction du pipeline scikit-learn (imputation, encodage catégoriel et standardisation)
6. Entraînement d'un modèle Random Forest avec pondération de classes
7. Évaluation rigoureuse : Précision, Rappel, F1-score, Matrice de confusion et Average Precision

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, precision_score, recall_score, f1_score

# 1. Chargement des données
df = pd.read_csv('bank_transactions.csv')
df.head()

In [ ]:
# 2. Proportion de fraudes et valeurs manquantes
print("Proportion de fraudes :")
print(df['is_fraud'].value_counts(normalize=True) * 100)

print("\nValeurs manquantes :")
print(df.isnull().sum())

In [ ]:
# 3 & 4. Exclusion des identifiants et séparation stratifiée
features = [c for c in df.columns if c not in ['transaction_id', 'account_id', 'is_fraud']]
X = df[features]
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

In [ ]:
# 5 & 6. Pipeline de prétraitement et modèle Random Forest
num_features = ['amount', 'hour', 'transactions_last_24h', 'average_amount_last_7d']
cat_features = ['country', 'merchant_category']

num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

pipeline.fit(X_train, y_train)

In [ ]:
# 7. Évaluation
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Matrice de confusion :\n", confusion_matrix(y_test, y_pred))
print("\nRapport de classification :\n", classification_report(y_test, y_pred, target_names=['Légitime', 'Fraude']))
print(f"Average Precision : {average_precision_score(y_test, y_prob):.4f}")